# ATRIUM RSE Assessment Task

This notebook contains code and discussion comments to complete the assessment task for the position of Research Software Engineer on the ATRIUM project. The tasks focused on interacting with the SSH Open Marketplace and alongside possible solutions to the tasks I've included some thoughts/insights to the marketplace API which I hope are useful.

## Issues with the SSH Open Marketplace OpenAPI Specification

Whilst it is possible to interact with the marketplace API using simple HTTP requests it makes more sense
to use the [published OpenAPI specification](https://marketplace-api.sshopencloud.eu/v3/api-docs) to generate
a custom client library which can validate both the responses from the API and any new data we wish to
submit to the marketplace.

The code in this notebook generates a custom OpenAPI client library using the
[OpenAPI Generator from OpenAPITools](https://github.com/OpenAPITools/openapi-generator).

Unfortunately generating a client library which validates the API responses highlighted a few problems with the
currently published documentation for the marketplace.

### Fields Containing Dates

The biggest problem I've discovered relating to the OpenAPI documentation of the marketplace is that every field
containing a date appears to be incorrectly defined. For example, the `registrationDate` property of the `UserDto`
component is defined as follows:

```
"registrationDate":{
    "pattern":"yyyy-MM-dd'T'HH:mm:ssZ",
    "type":"string",
    "example":"2000-02-29T20:02:00+0000"
}
```

At first glance this looks sensible, and suggests that the date must conform to the `yyyy-MM-dd'T'HH:mm:ssZ`
format. The problem though, is that according to the OpenAPI specification the `pattern` field should be used
as a regular expression to validate the value of the string field. In this case, that means that only the
pattern string itself would, I beleive, be a valid value for this field. According to the OpenAPI specification
the correct way to specify dates is by using the  `format` field. In this case that would result in the following:

```
"registrationDate":{
    "format": "date-time",
    "type":"string",
    "example":"2000-02-29T20:02:00+0000"
}
```

The OpenAPI specification states that valid values for this field must conform to `date-time` as defined by
[RFC3339](https://datatracker.ietf.org/doc/html/rfc3339#section-5.6), which includes the expected
`yyyy-MM-dd'T'HH:mm:ssZ` format.

I assume this was missed because the marketplace OpenAPI description, as currently published, is syntactically
valid just not logically correct.

In order to generate a useable client library I've produced [an updated version of the specification](api-docs.json)
which now correctly specifies dates.

### Required Fields

According to the [Metadata Guidelines](https://marketplace.sshopencloud.eu/contribute/metadata-guidelines)
for the marketplace both a `label` and `description` are mandatory for every resource type. The provided
OpenAPI description, however, states only that `label` is a required field meaning that a JSON object such as

```
{
    "label": "some label text"
}
```

would be a valid payload for creating a publication, dataset, training material, or workflow. At a minimum
(to match the guidelines) the `description` field should also be marked as required, but it is likely that
other fields should also be marked as required for many of the resource types, although without access to
the software behind the marketplace it is not possible to know which fields are truly required for the
creation of each resource type to succeed.

### Minimum/Maximum Values

Note there is a mistake in the error response for `/api/tools-services`. If you set the `page` param to 0 then the response is

```
{
  "timestamp": "2026-04-23 09:47:54",
  "status": 400,
  "error": "Page index must not be less than zero"
}
```

The message should more properly read "Page index must not be less than zero". Note that no where in the documentation (that I can find) does it state that the page numbers start at 1. Whilst starting at 1 for the UI makes sense, most people consuming the REST API would probably assume that the numbers count from zero.

Similarly I discovered the maximum number of results per page is 100, again this is not documented anywhere

Best way to document this would probably be to set a minimum value in the OpenAPI description of the `page` param:

```
{
  "name": "page",
  "in": "query",
  "required": false,
  "schema": {
    "type": "integer",
    "format": "int32",
    "minimum": 1
  }
}
```

This would have the added affect of clients not even making the call if the value was 0 as it wouldn't validate.

## Initial Environment Setup

This section of the notebook focuses on setting up the environment so that we can easily communicate with the marketplace.
None of the code in this section is specific to any of the assessment options.

We could have included the dependencies we need in a separate `requirements.txt` file but I've chosen instead to generate the file from within the notebook so that the dependencies we are using are made explicit and easy to view without having to switch to look at a separate file

In [ ]:
%%writefile requirements.txt

# this is used to generate the custom client library to aid in
# accessing the marketplace REST API
openapi-generator-cli==7.21.0

# only needed for the data ananysis in the Option B task
pandas==3.0.2

We can now install the dependencies, generate the custom client library, and then install that into the notebook environment

In [ ]:
# install all the dependencies we need for the rest of this notebook
%pip install -r requirements.txt

# generate a custom client library for working with the SSH Open Marketplace.
#
# note that we are using a local copy of the OpenAPI description for the marketplace
# due to the issue around dates. Once the official version has been updated this
# command couild be updated to use https://marketplace-api.sshopencloud.eu/v3/api-docs
! openapi-generator-cli generate -g python -i ./api-docs.json -o marketplace-client

# install the custom client library into the environment ready for us to use
%pip install ./marketplace-client

With the required dependencies installed we can now import and configure the custom client library

In [ ]:
import openapi_client

# By default the custom client library we created defaults to
# connecting to https://marketplace-api.sshopencloud.eu
# Should this need to be overriden a host parameter can be
# passed to the Configuration constructor. Note that this
# is where you would also specify authentication details etc.
configuration = openapi_client.Configuration()

Before we start on the actual tasks let's also import some utility functions that just aid us in displaying the results

In [ ]:
# this lets us nicely format output cells to show JSON or Markdown
from IPython.display import JSON, Markdown, display

## Option A – Item Type Conversion

The task here is to take an item that has been created in the wrong category and map the data into a JSON object which could be used to re-create the object in the right category. I've chosen to map a Publication to a Dataset both as that was the example given in the task description but also as it seems the most likely real world mis-categorization. 

### Retrieve a Publication from the Marketplace

We start by specifying the persistent ID of the publication we want to convert. This can be found either from the `persistentId` field of a publication JSON record or from the end of the URL when viewed through the marketplace web application.

In [ ]:
# As an example lets use the following publication
# https://marketplace.sshopencloud.eu/publication/cy3JNJ
# which is entitled:
#   The competitive esports physiological, affective, and video dataset
publication_id = "cy3JNJ"

Once we have the ID of a publication we can use the client library to retrieve it from the marketplace via the API

In [ ]:

# Enter a context with a correctly configured instance of the API client
with openapi_client.ApiClient(configuration) as api_client:
    
    # get the section of the API that handles publications
    # i.e. this gives access to everything in the spec tagged as
    # publication-controller which is essentially all the endpoints
    # under /api/publications/  
    publication_api = openapi_client.PublicationControllerApi(api_client)
    
    # retrieve the specific publication we requested above
    # Note that the object returned is an instance of PublicationDto
    # as described in the OpenAPI specification document.
    publication = publication_api.get_publication(publication_id);

# display the retrieved JSON object so we can take a look at what
# we get back. Obviously not necessary in a real world system but
# useful here for debugging and sanity checking etc.
JSON(publication.to_dict(), expanded = True)

### Convert the Publication into a Dataset

The `/api/datasets` endpoint used to create a new dataset within the marketplace expects us to POST a `DatasetCore` instance. Fortunately the client library makes it easy to create a `DatasetCore` instance by copying the values from a Python dict provided by the `PublicationDto` instance.

In [ ]:

# If we want to create a new dataset from the publication then we need
# to complete an instance of DatasetCore that we could then POST to
# /api/datasets (using DatasetControllerAPI.create_dataset())
from openapi_client.models import DatasetCore

# copying the data from the publication is easy as both objects support
# being converted to, or created from, a dict, so we convert the publication
# to a dict and then create a dataset from that dict.
#
# Because of the way this works inside the client library it recurses through
# nested objects copying the values that are needed. This coupled with the fact
# that the *Core components are all proper subsets of their *Dto counterparts
# means that while we ignore some information in the publication we should have
# everything we need to construct the dataset
dataset = DatasetCore.from_dict(publication.to_dict())

As an initial sanity check we can check which elements were not copied from the publicaiton into the new dataset instance

In [ ]:
# turn the keys from both the publication and dataset into sets and then turn the
# difference between the two sets into a single string that we can format as
# a list in Markdown
Markdown("- "+"\n- ".join(publication.to_dict().keys() - dataset.to_dict().keys()))

Given the dataset has not yet been created (we are just building the payload for the creation request) it makes sense that `id`, `persistentId`, `status`, and `lastInfoUpdate` are not copied across. Also we do not need to provide a `category` field as this will be automaticaly set to `dataset` by the marketplace.

That just leaves the `informationContributor` field. My understanding of this field is that it records the last person to edit the record. Given the dataset does not exist it makes sense that this field is not copied over, although it is unfortunate that creating a dataset in this way will sever the edit history at this point (assuming the original publication instance is deleted). One option would be to add the user into the list of contributors, but as the `informationContributor` is often a marketplace moderator, and not related to the creration of the original dataset, this does not seem a sensible option.

The [Metadata Guidelines](https://marketplace.sshopencloud.eu/contribute/metadata-guidelines) show that the main difference between a publication and a dataset relates to bibliographic metadata. All fields are recommended for publications but only `publisher` and `year` are recommended for datasets. It would be trivial to remove the other properties (such as `publication-type` etc.) but as providing them is not explicitly disallowed it makes more sense to leave them in place; someone thought they were important enough to provide originally and a moderator could always choose to remove them when accepting the new dataset record.

Unfortunately the [Metadata Guidelines](https://marketplace.sshopencloud.eu/contribute/metadata-guidelines) do not provide information on every field present in either the `PublicationDto` or `DatasetCore` components which does leave some ambiguity in the values of certain fields, especially `dateCreated` and `dateLastUpdated`. I assume that these both refer to the creation of the actual record within the marketplace and not to the creation or modified date of the publication or dataset being described. Assuming that is the case then it is rather odd to include them in the request to create a new dataset as I would assume the server would record the time it added the record. Given we are essentially converting one record to another, however, I'm tempted to provide both dates leaving the `dateCreated` as that given in the publication record but using the current time for the `dateLastUpdated` field.


In [ ]:
from datetime import datetime

# update the dateLastUpdated prioperty to the current time
#
# note I've used a specific format pattern here to ensure the value matches the pattern given
# in the original OpenAPI docs for the marketplace (yyyy-MM-dd'T'HH:mm:ssZ) otherwise the
# default would be to include milliseconds as well. That would validate against the modified
# specification (i.e. it's a valid date as defined in RFC3339) in use with the client library
# but may cause an issue when submitted to the marketplace so this is safer.
dataset.date_last_updated = datetime.now().strftime("%Y-%m-%dT%H:%M:%S%Z")

One extra piece of metadata that would be potentially useful to add would be an explicit link back to the publication that the dataset was created from. The most obvious way of doing this would be with a `see-also` metadata property:

In [ ]:
# adding a new property requires creating instances of the PropertyCore and
# PropertyTypeId components as defined in the marketplace OpenAPI spec so
# we import those models from the custom client library we generated
from openapi_client.models import PropertyCore, PropertyTypeId

# append the new "see-also" property to the list of existing properties
dataset.properties.append(
    PropertyCore(
        type = PropertyTypeId(code = "see-also"),

        # I think a link to the marketplace web app makes more sense
        # here than a link to the API endpoint that would return JSON
        value = f"https://marketplace.sshopencloud.eu/publication/{publication_id}"
    )
)

### Produce the JSON Payload

With the mapping from publication to dataset completed we could now POST this to the `/api/datasets` endpoint to create the new entry in the marketplace. Given we don't want to do this though we will simply display the JSON payload instead

In [ ]:
# converting the DatasetCore instance to a dict and then using the JSON
# IPython display function gives a nice human readbale form of the payload.
JSON(dataset.to_dict(), expanded = True)

### Final Thoughts on the Task

The sections of the marketplace API used for this task are easy to use and intuitive. More detail of some properties, either in the guidelines or within the OpenAPI description, would be useful especially for fields such as `dateCreated` and `dateLastModified`. My feeling is that some of these fields are either already redundant (i.e. the marketplace will ignore their value) or should be removed and populated automatically (i.e. the marketplace should populate these values itself) -- being able to alter date based info used for tracking the edit history etc. does not seem like a sensible approach, although the danger of abuse is limited given you need to be authenticated in order to add a new dataset.

## Option B - Duplicate Detection

### Retrieve all the Tools/Services from the Marketplace

In [ ]:

tools = []

# Enter a context with a correctly configured instance of the API client
with openapi_client.ApiClient(configuration) as api_client:
    
    # get the section of the API that handles tools and services
    # i.e. this gives access to everything in the spec tagged as
    # tool-controller which is essentially all the endpoints
    # under /api/tools-services/  
    tools_api = openapi_client.ToolControllerApi(api_client)
    
    # for some strange reason the API counts pages from 1 not 0
    # so we want to start by getting page 1
    page = 1
    
    # we will assume there is only one page of results, and then
    # we can update this once the first request has completed.
    pages = 1

    while page <= pages:
        # while there are still pages of results to collect...

        # give some feedback as this can take a while
        print(f"retrieving page {page} of {pages if pages != 1 else "?"}")
        
        # get the next page of tools from the marketplace
        response = tools_api.get_tools(page=page, perpage=100)
        
        # update the total number of pages so we never
        # try and ask for more than there are left
        pages = response.pages
        
        for tool in response.tools:
            # convert each tool response we got back into a dict
            # and then add it to the list of tools we are collecting
            tools.append(tool.to_dict())
        
        # get ready to collect the next page of results
        page = page + 1 

# how many tools are there that we need to process?
print(f"Collected information on {len(tools)} tools or services")


### Find Potential Duplicates via `accessibleAt`

In [251]:
# we are going to use the data handling and aggregation functions
# from DataFrame which is part of the pandas library so we need to
# import that here before we do anything else
import pandas as pd

# create a DataFrame where each row contains just the persistentId
# and accessiableAt data from the tools info we collected from the
# marketplace
df = pd.DataFrame(tools, columns=["persistentId", "accessibleAt"])

# the accessibleAt filed is an array of zero or more URLs. This
# explodes the list so that each row is a URL and persistentId pair
df = df.explode("accessibleAt").reset_index(drop=True)

# if the accessibleAt array was empty then we end up with a row
# where the value is now null, and we can easily filter those out
df = df[df["accessibleAt"].notnull()]

# we now group the data based on the accessibleAt field so we end
# up with essentially the opposite of what we started with, where
# each row is now a URL and a list of persistentId values
df = df.groupby(['accessibleAt']).agg(pids=('persistentId', list)).reset_index()

# add a new count column which is the number of persistentId values
df["count"] = df['pids'].str.len()

# filter the dataframe to only keep rows in which the same URL was
# associated with multiple tools
df = df[(df['count'] > 1)]

# sort the remaining values so we look at those URLs with the least
# tools first -- a URL with many tools is likely to be a suite of
# tools all available from the same URL rather than duplications
df = df.sort_values(by=['count'], ascending=True)

# display a summary (if for no other reason than it makes it clear
# that this cell has been run, given how fast it completes).
Markdown(f"There are {df.shape[0]} URLs which are associated with one or more tools. These URLs relate to {df["count"].sum()} tools or services.")

There are 64 URLs which are associated with one or more tools. These URLs relate to 176 tools or services.

In [ ]:

# create a new dataframe that essentially works as a mapping from the
# persistent ID of a tool to it's human readable label so that we can
# display these as part of the output
pidLabels = pd.DataFrame(tools, columns=["label", "persistentId"])

# given we are going to try a number of steps to find possible duplicates
# let's define a function we can use to pretty print each set so we can
# use it multiple times without having to simply copy-and-paste it across
# the notebook...

def displayDuplicates(duplicates: pd.DataFrame, labels: pd.DataFrame):
    """Displays each URL and the list of possibly duplicated tools
       as a Markdown list for nicer display in the notebook For each
       URL the outlook looks like

       <URL> is the URL for the following <X> tools:
       - <label of tool 1>
       - <label of tool 2>

       Where the labels are also links back to the marketplace

       Parameters
       ----------

       duplicates: a pandas DataFrame where each row contains a URL and
            a list of persistent IDs (in the columns 'assesibleAt' and 'pids'
            respectively).
        labels: a pandas Dataframe that maps 'persistentId' fields to the
            'label' field for the tool

    """

    for row in duplicates.itertuples():
        # for each URL that we think is associated with duplicated tools,
        # build up and display a nicelty formated Markdown block to show
        # the URL and possible duplicated tools.
        display(Markdown(f"""[{row.accessibleAt}]({row.accessibleAt}) is the URL for the following {len(row.pids)} tools:
        \n- {"\n- ".join([f"[{labels[labels["persistentId"] == p].iloc[0]["label"]}](https://marketplace.sshopencloud.eu/tool-or-service/{p})" for p in row.pids])}"""))


# now we have the function defined we can display the potential
# duplicates we think we have found from the initial analysis
displayDuplicates(df, pidLabels)

### Try to Reduce False Positives via Text Similarity

In [ ]:

from itertools import combinations
import difflib

scores = []

for row in df.itertuples():
    selected = pidLabels[pidLabels["persistentId"].isin(row.pids)]["label"]
    score = 0.0
    pairs = list(combinations(selected,2))
    for [p1, p2] in pairs: 
        score = score + difflib.SequenceMatcher(None, p1.lower(), p2.lower()).ratio()

    score = score / len(pairs)

    scores.append(score)

df["score"] = scores

# filter the dataframe to only keep rows in which the same URL was
# associated with multiple tools
filtered = df[(df['score'] > 0.6)]

filtered = filtered.sort_values(by=['score'], ascending=False)

# display a summary (if for no other reason than it makes it clear
# that this cell has been run, given how fast it completes).
display(Markdown(f"""There are {filtered.shape[0]} URLs which are associated with one or more tools.
                 These URLs relate to {filtered["count"].sum()} tools or services."""))

In [ ]:
displayDuplicates(filtered, pidLabels)

### Final Thoughts on the Task